In [1]:
from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path(r"D:\Sami Data Set")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
TRAIN_DIR = SPLIT_DIR / "train"
TEST_DIR = SPLIT_DIR / "test"

DOCUMENTATION_DIR = PROJECT_DIR / "documentation"
SPLIT_INDEX_DIR = DOCUMENTATION_DIR / "split_indices"

TRAIN_DIR.mkdir(parents=True, exist_ok=True)
TEST_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_INDEX_DIR.mkdir(parents=True, exist_ok=True)

processed_files = sorted(
    PROCESSED_DIR.glob("*_processed.parquet")
)

if not processed_files:
    raise FileNotFoundError(
        f"No processed files found in:\n{PROCESSED_DIR}"
    )

print("Processed files found:", len(processed_files))

Processed files found: 10


In [2]:
TEST_SIZE = 0.20
TRAIN_SIZE = 1 - TEST_SIZE
RANDOM_SEED = 42

print(f"Training proportion: {TRAIN_SIZE:.0%}")
print(f"Testing proportion: {TEST_SIZE:.0%}")
print(f"Random seed: {RANDOM_SEED}")

Training proportion: 80%
Testing proportion: 20%
Random seed: 42


In [3]:
split_records = []

for file_number, file_path in enumerate(
    processed_files,
    start=1
):
    print(
        f"\n[{file_number}/{len(processed_files)}] "
        f"Splitting {file_path.name}",
        flush=True
    )

    try:
        df = pd.read_parquet(file_path)

        required_columns = {
            "binary_label",
            "original_attack_label",
            "source_file_id",
            "source_row_id"
        }

        missing_columns = required_columns.difference(df.columns)

        if missing_columns:
            raise KeyError(
                f"Required columns missing: {missing_columns}"
            )

        binary_values = df["binary_label"].to_numpy()

        # Empty test mask: every record initially belongs to training
        test_mask = np.zeros(len(df), dtype=bool)

        class_split_records = []

        for class_value in [0, 1]:
            class_positions = np.flatnonzero(
                binary_values == class_value
            )

            if len(class_positions) == 0:
                continue

            class_seed = (
                RANDOM_SEED
                + file_number * 100
                + class_value
            )

            rng = np.random.default_rng(class_seed)

            test_count = int(
                round(len(class_positions) * TEST_SIZE)
            )

            selected_test_positions = rng.choice(
                class_positions,
                size=test_count,
                replace=False
            )

            test_mask[selected_test_positions] = True

            class_split_records.append({
                "binary_label": class_value,
                "total_count": len(class_positions),
                "test_count": test_count,
                "train_count": (
                    len(class_positions) - test_count
                )
            })

        # Save frozen test row identifiers for reproducibility
        test_source_row_ids = df.loc[
            test_mask,
            "source_row_id"
        ].to_numpy(dtype=np.int64)

        np.save(
            SPLIT_INDEX_DIR
            / f"{file_path.stem}_test_source_row_ids.npy",
            test_source_row_ids
        )

        # Save test set first
        test_df = df.loc[test_mask].copy()

        test_output = (
            TEST_DIR
            / file_path.name.replace(
                "_processed.parquet",
                "_test.parquet"
            )
        )

        test_df.to_parquet(
            test_output,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        test_rows = len(test_df)
        test_benign = int(
            (test_df["binary_label"] == 0).sum()
        )
        test_malicious = int(
            (test_df["binary_label"] == 1).sum()
        )

        del test_df
        gc.collect()

        # Save training set
        train_df = df.loc[~test_mask].copy()

        train_output = (
            TRAIN_DIR
            / file_path.name.replace(
                "_processed.parquet",
                "_train.parquet"
            )
        )

        train_df.to_parquet(
            train_output,
            engine="pyarrow",
            compression="snappy",
            index=False
        )

        train_rows = len(train_df)
        train_benign = int(
            (train_df["binary_label"] == 0).sum()
        )
        train_malicious = int(
            (train_df["binary_label"] == 1).sum()
        )

        split_records.append({
            "source_file": file_path.name,
            "train_file": train_output.name,
            "test_file": test_output.name,
            "total_rows": len(df),
            "train_rows": train_rows,
            "test_rows": test_rows,
            "train_benign": train_benign,
            "train_malicious": train_malicious,
            "test_benign": test_benign,
            "test_malicious": test_malicious,
            "train_percentage": round(
                train_rows / len(df) * 100,
                4
            ),
            "test_percentage": round(
                test_rows / len(df) * 100,
                4
            ),
            "status": "Completed",
            "error": ""
        })

        print(
            f"Completed | Train: {train_rows:,} | "
            f"Test: {test_rows:,}",
            flush=True
        )

        del train_df
        del df
        del test_mask
        del test_source_row_ids
        gc.collect()

    except Exception as error:
        split_records.append({
            "source_file": file_path.name,
            "train_file": "",
            "test_file": "",
            "total_rows": None,
            "train_rows": None,
            "test_rows": None,
            "train_benign": None,
            "train_malicious": None,
            "test_benign": None,
            "test_malicious": None,
            "train_percentage": None,
            "test_percentage": None,
            "status": "Error",
            "error": str(error)
        })

        print("Error:", error, flush=True)

split_summary = pd.DataFrame(split_records)

display(split_summary)


[1/10] Splitting Botnet-Friday-02-03-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 617,270 | Test: 154,317

[2/10] Splitting Bruteforce-Wednesday-14-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 495,477 | Test: 123,869

[3/10] Splitting DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 763,877 | Test: 190,969

[4/10] Splitting DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 449,117 | Test: 112,279

[5/10] Splitting DoS1-Thursday-15-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 635,849 | Test: 158,963

[6/10] Splitting DoS2-Friday-16-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 473,498 | Test: 118,375

[7/10] Splitting Infil1-Wednesday-28-02-2018_TrafficForML_CICFlowMeter_processed.parquet
Completed | Train: 365,498 | Test: 91,375

[8/10] Splitting Infil2-Thursday-01-03-2018_TrafficForML_CICFlowMeter_processed.

,source_file,train_file,test_file,total_rows,train_rows,test_rows,train_benign,train_malicious,test_benign,test_malicious,train_percentage,test_percentage,status,error
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,771587,617270,154317,501642,115628,125410,28907,80.0001,19.9999,Completed,
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,619346,495477,123869,420196,75281,105049,18820,80.0000,20.0000,Completed,
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,954846,763877,190969,303586,460291,75896,115073,80.0000,20.0000,Completed,
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,561396,449117,112279,288644,160473,72161,40118,80.0000,20.0000,Completed,
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,794812,635849,158963,594798,41051,148700,10263,79.9999,20.0001,Completed,
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,591873,473498,118375,357295,116203,89324,29051,79.9999,20.0001,Completed,
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,456873,365498,91375,320339,45159,80085,11290,79.9999,20.0001,Completed,
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,249170,199336,49834,149709,49627,37427,12407,80.0000,20.0000,Completed,
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,830224,664179,166045,663906,273,165977,68,80.0000,20.0000,Completed,
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,829405,663524,165881,663091,433,165773,108,80.0000,20.0000,Completed,


In [4]:
split_summary_path = (
    DOCUMENTATION_DIR / "train_test_split_summary.csv"
)

split_summary.to_csv(
    split_summary_path,
    index=False
)

split_configuration = {
    "split_method": (
        "Stratified random split within each source file"
    ),
    "training_proportion": TRAIN_SIZE,
    "testing_proportion": TEST_SIZE,
    "random_seed": RANDOM_SEED,
    "stratification_variable": "binary_label",
    "test_set_status": (
        "Frozen and excluded from feature selection, "
        "scaling, balancing and tuning"
    )
}

with open(
    DOCUMENTATION_DIR / "split_configuration.json",
    "w",
    encoding="utf-8"
) as configuration_file:
    json.dump(
        split_configuration,
        configuration_file,
        indent=4
    )

print("Split summary saved to:")
print(split_summary_path)

Split summary saved to:
D:\Sami Data Set\documentation\train_test_split_summary.csv


In [5]:
total_rows = int(split_summary["total_rows"].sum())
total_train = int(split_summary["train_rows"].sum())
total_test = int(split_summary["test_rows"].sum())

train_benign = int(
    split_summary["train_benign"].sum()
)

train_malicious = int(
    split_summary["train_malicious"].sum()
)

test_benign = int(
    split_summary["test_benign"].sum()
)

test_malicious = int(
    split_summary["test_malicious"].sum()
)

overall_split_summary = pd.DataFrame([
    {
        "split": "Training",
        "benign": train_benign,
        "malicious": train_malicious,
        "total": total_train,
        "percentage_of_dataset": round(
            total_train / total_rows * 100,
            4
        )
    },
    {
        "split": "Testing",
        "benign": test_benign,
        "malicious": test_malicious,
        "total": total_test,
        "percentage_of_dataset": round(
            total_test / total_rows * 100,
            4
        )
    }
])

display(overall_split_summary)

print("Original records:", f"{total_rows:,}")
print("Training records:", f"{total_train:,}")
print("Testing records:", f"{total_test:,}")
print(
    "Train + test equals original:",
    total_train + total_test == total_rows
)

overall_split_summary.to_csv(
    DOCUMENTATION_DIR / "overall_split_summary.csv",
    index=False
)

,split,benign,malicious,total,percentage_of_dataset
0,Training,4263206,1064419,5327625,80.0
1,Testing,1065802,266105,1331907,20.0


Original records: 6,659,532
Training records: 5,327,625
Testing records: 1,331,907
Train + test equals original: True


In [6]:
train_files = sorted(TRAIN_DIR.glob("*_train.parquet"))
test_files = sorted(TEST_DIR.glob("*_test.parquet"))

overlap_records = []

for train_file, test_file in zip(
    train_files,
    test_files
):
    train_ids = pd.read_parquet(
        train_file,
        columns=["source_file_id", "source_row_id"]
    )

    test_ids = pd.read_parquet(
        test_file,
        columns=["source_file_id", "source_row_id"]
    )

    train_keys = pd.MultiIndex.from_frame(train_ids)
    test_keys = pd.MultiIndex.from_frame(test_ids)

    overlap_count = len(
        train_keys.intersection(test_keys)
    )

    overlap_records.append({
        "train_file": train_file.name,
        "test_file": test_file.name,
        "overlapping_rows": overlap_count
    })

    del train_ids
    del test_ids
    del train_keys
    del test_keys
    gc.collect()

overlap_summary = pd.DataFrame(overlap_records)

display(overlap_summary)

print(
    "Total overlapping rows:",
    overlap_summary["overlapping_rows"].sum()
)

overlap_summary.to_csv(
    DOCUMENTATION_DIR / "train_test_overlap_check.csv",
    index=False
)


,train_file,test_file,overlapping_rows
0,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,Botnet-Friday-02-03-2018_TrafficForML_CICFlowM...,0
1,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,Bruteforce-Wednesday-14-02-2018_TrafficForML_C...,0
2,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,DDoS1-Tuesday-20-02-2018_TrafficForML_CICFlowM...,0
3,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,DDoS2-Wednesday-21-02-2018_TrafficForML_CICFlo...,0
4,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,DoS1-Thursday-15-02-2018_TrafficForML_CICFlowM...,0
5,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,DoS2-Friday-16-02-2018_TrafficForML_CICFlowMet...,0
6,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,Infil1-Wednesday-28-02-2018_TrafficForML_CICFl...,0
7,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,Infil2-Thursday-01-03-2018_TrafficForML_CICFlo...,0
8,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,Web1-Thursday-22-02-2018_TrafficForML_CICFlowM...,0
9,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,Web2-Friday-23-02-2018_TrafficForML_CICFlowMet...,0


Total overlapping rows: 0


## Train–Test Split Conclusion

The processed dataset was divided using a fixed 80:20 stratified
train–test split. Stratification was performed separately within each
source file using the binary traffic label, ensuring that benign and
malicious records from each traffic scenario were represented in both
sets.

A fixed random seed of 42 was used, and the selected test-row
identifiers were saved to support reproducibility. The training and
test records were written to separate directories, and no overlap was
identified between the two sets.

The test set is now frozen. It will not be used during feature
selection, scaling, class balancing, model tuning or model selection.
All subsequent development decisions will be based on the training
data only.